In [153]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# visualization
import seaborn as sns
import matplotlib.pyplot as plt


from pyspark.ml.feature import VectorAssembler

In [154]:
LABEL_COLUMN='price'
SEED=42

spark = SparkSession.builder \
    .appName("Airbnb") \
    .getOrCreate()

raw_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("mode", "PERMISSIVE")
    .parquet("./total_data.parquet")
)

raw_df.createOrReplaceTempView('raw_listing')

In [155]:
# as the dataset is has a lot of columns, we did an analysis and selected the most 
# good appearing to the prediction. 

selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification, 
        month,
        security_deposit,
        cleaning_fee
    FROM raw_listing
""")

selected_df.createOrReplaceTempView("selected_df")

## Initial Parsing

In [156]:
selected_df.show()

+-------------------+-------------------+-----------------+-------------------+------------------+------------+---------+--------+----+--------------------+---------+-----------------------------+--------------------------------+-----+----------------+------------+
|           latitude|          longitude|host_is_superhost|host_listings_count|     property_type|accommodates|bathrooms|bedrooms|beds|           amenities|    price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+-------------------+-------------------+-----------------+-------------------+------------------+------------+---------+--------+----+--------------------+---------+-----------------------------+--------------------------------+-----+----------------+------------+
|-22.965919031411442| -43.17896230586568|                f|                2.0|       Condominium|           5|      1.0|     2.0| 2.0|{TV,"Cable TV",In...|  $307.00|                            f|      

Shape of the dataset:

In [157]:
# change this to pure SQL
print(f'Rows: {selected_df.count()}, Columns: {len(selected_df.columns)}')

Rows: 784122, Columns: 16


Checking for Null's:

In [158]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|security_deposit|cleaning_fee|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+----------------+------------+
|       1|        1|              386|                386|            1|           1|     1494|     776|2335|        1|    1|                            2|                               2|    2|          361064|      269336|
+--------+---------+-----------------+-------------------+-------------+------------+---------+-----

As we can see there's a lot of missing values in the `security_deposit` and `cleaning_fee` columns.

Dropping them from the database seems to be a good decision.

In [159]:
rows_before_drop = selected_df.count()
selected_df = selected_df.drop('security_deposit', 'cleaning_fee')

In [160]:
selected_df = selected_df.dropna()
print(f"({selected_df.count()}, {len(selected_df.columns)}) - {rows_before_drop - selected_df.count()} rows dropped")

(780035, 14) - 4087 rows dropped


Checking if there is any missing left:

In [161]:
selected_df.createOrReplaceTempView("selected_df")
null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()


+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



In [162]:
selected_df.createOrReplaceTempView('selected_df')

### Checking data types

In [163]:
selected_df.printSchema()

root
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_listings_count: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- accommodates: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- beds: string (nullable = true)
 |-- amenities: string (nullable = true)
 |-- price: string (nullable = true)
 |-- require_guest_profile_picture: string (nullable = true)
 |-- require_guest_phone_verification: string (nullable = true)
 |-- month: string (nullable = true)



TODO: A brief explanation of what need to be changed

### Checking data types

In [164]:
selected_df = spark.sql("""
    SELECT
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        TRY_CAST(host_is_superhost AS BOOLEAN) AS host_is_superhost,
        TRY_CAST(TRY_CAST(host_listings_count AS DOUBLE) AS INT) AS host_listings_count,
        property_type,
        TRY_CAST(TRY_CAST(accommodates AS DOUBLE) AS INT) AS accommodates,
        TRY_CAST(TRY_CAST(bathrooms AS DOUBLE) AS INT) AS bathrooms,
        TRY_CAST(TRY_CAST(bedrooms AS DOUBLE) AS INT) AS bedrooms,
        TRY_CAST(TRY_CAST(beds AS DOUBLE) AS INT) AS beds,
        amenities,
        TRY_CAST(REGEXP_REPLACE(price, '[\\\\$,]', '') AS DOUBLE) AS price,
        TRY_CAST(require_guest_profile_picture AS BOOLEAN) AS require_guest_profile_picture,
        TRY_CAST(require_guest_phone_verification AS BOOLEAN) AS require_guest_phone_verification, 
        TRY_CAST(month AS INT) AS month
    FROM selected_df
""")

selected_df.createOrReplaceTempView("selected_df")

null_count_expr = ",\n    ".join([
    f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
    for c in selected_df.columns
])
spark.sql(f"SELECT\n    {null_count_expr}\nFROM selected_df").show()

+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|latitude|longitude|host_is_superhost|host_listings_count|property_type|accommodates|bathrooms|bedrooms|beds|amenities|price|require_guest_profile_picture|require_guest_phone_verification|month|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+
|       0|        0|                0|                  0|            0|           0|        0|       0|   0|        0|    0|                            0|                               0|    0|
+--------+---------+-----------------+-------------------+-------------+------------+---------+--------+----+---------+-----+-----------------------------+--------------------------------+-----+



## EDA + Data Cleaning

### Dealing with outliers

In [165]:
def get_max_fence(column):
    qt = selected_df.approxQuantile(column, [0.25,0.75], 0.01)
    q1 = qt[0]
    upper = qt[1]
    iqr = upper - q1
    max_fence = upper + 1.5 * iqr
    return max_fence

#### `host_listing_count`

In [166]:
listing_count_max_fence = get_max_fence('host_listings_count')
print(listing_count_max_fence)

6.0


In [167]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE host_listings_count <= {listing_count_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

99528 rows were deleted.


In [168]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        CASE WHEN host_listings_count = 0.0 THEN 1.0 ELSE host_listings_count END AS host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `price`

In [169]:
price_max_fence = get_max_fence('price')
print(price_max_fence)

1274.5


In [170]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE price <= {price_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

66373 rows were deleted.


#### `property_type`

In [171]:
categories_to_append = ('Aparthotel', 'Earth house', 'Chalet', 'Cottage', 'Tiny house',
                        'Boutique hotel', 'Hotel', 'Casa particular (Cuba)', 'Bungalow',
                        'Nature lodge', 'Cabin', 'Castle', 'Treehouse', 'Island', 'Boat', 'Tent',
                        'Resort', 'Hut', 'Campsite', 'Barn', 'Dorm', 'Camper/RV', 'Farm stay', 'Yurt',
                        'Tipi', 'Pension (South Korea)', 'Dome house', 'Igloo', 'Casa particular',
                        'Houseboat', 'Lighthouse', 'Plane', 'Train', 'Parking Space')

category_list_sql = ", ".join([f"'" + c.replace("'", "''") + "'" for c in categories_to_append])
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        CASE WHEN property_type IN ({category_list_sql}) THEN 'Other' ELSE property_type END AS property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month
    FROM selected_df
""")

#### `beds`

In [172]:
beds_max_fence = get_max_fence('beds')
print(beds_max_fence)

6.0


In [173]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE beds <= {beds_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

13846 rows were deleted.


#### `amenities`

In [174]:
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql("""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        amenities,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        SIZE(SPLIT(amenities, ',')) + 1 AS n_amenities
    FROM selected_df
""")
selected_df.createOrReplaceTempView('selected_df')
mode_value = spark.sql("""
    SELECT n_amenities
    FROM selected_df
    WHERE amenities <> '{}'
    GROUP BY n_amenities
    ORDER BY COUNT(*) DESC
    LIMIT 1
""").first()['n_amenities']
selected_df = spark.sql(f"""
    SELECT
        latitude,
        longitude,
        host_is_superhost,
        host_listings_count,
        property_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        price,
        require_guest_profile_picture,
        require_guest_phone_verification,
        month,
        CASE WHEN amenities = '{{}}' THEN {mode_value} ELSE n_amenities END AS n_amenities
    FROM selected_df
""")

In [175]:
amenities_max_fence = get_max_fence('n_amenities')
print(amenities_max_fence)

40.5


In [176]:
rows_before = selected_df.count()
selected_df.createOrReplaceTempView('selected_df')
selected_df = spark.sql(f"""
    SELECT *
    FROM selected_df
    WHERE n_amenities <= {amenities_max_fence}
""")
print(f"{rows_before - selected_df.count()} rows were deleted.")

22771 rows were deleted.


## Encoding

In [177]:
selected_df.createOrReplaceTempView('selected_df')
df_label_encoder = spark.sql("""
    WITH property_type_index AS (
        SELECT
            property_type,
            ROW_NUMBER() OVER (ORDER BY cnt DESC, property_type ASC) - 1 AS property_type_encoded
        FROM (
            SELECT property_type, COUNT(*) AS cnt
            FROM selected_df
            GROUP BY property_type
        )
    )
    SELECT
        s.latitude,
        s.longitude,
        CASE
            WHEN s.host_is_superhost IS TRUE THEN 1
            WHEN s.host_is_superhost IS FALSE THEN 0
            ELSE CAST(s.host_is_superhost AS INT)
        END AS host_is_superhost,
        s.host_listings_count,
        p.property_type_encoded AS property_type,
        s.accommodates,
        s.bathrooms,
        s.bedrooms,
        s.beds,
        s.price,
        CASE
            WHEN s.require_guest_profile_picture IS TRUE THEN 1
            WHEN s.require_guest_profile_picture IS FALSE THEN 0
            ELSE CAST(s.require_guest_profile_picture AS INT)
        END AS require_guest_profile_picture,
        CASE
            WHEN s.require_guest_phone_verification IS TRUE THEN 1
            WHEN s.require_guest_phone_verification IS FALSE THEN 0
            ELSE CAST(s.require_guest_phone_verification AS INT)
        END AS require_guest_phone_verification,
        s.month,
        s.n_amenities
    FROM selected_df s
    LEFT JOIN property_type_index p
        ON s.property_type = p.property_type
""")
print('Columns encoded')

Columns encoded


## Model

In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
models = [
    (
        "Linear Regression",
        LinearRegression(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxIter=50,
            regParam=0.1,
            elasticNetParam=0.0
        )
    ),
    (
        "Decision Tree Regressor",
        DecisionTreeRegressor(
            labelCol=LABEL_COLUMN,
            featuresCol="features",
            maxDepth=30,
            minInstancesPerNode=20,
            minInfoGain=0.0,
            maxBins=64,
            seed=SEED
        )
    ),
]

In [179]:
feature_cols = [
    "host_is_superhost",
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "month",
    "property_type",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "n_amenities"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(df_label_encoder)

train_prepared, test_prepared = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

print(f"Linhas no Treino: {train_prepared.count()}")
print(f"Linhas no Teste: {test_prepared.count()}")

26/05/10 17:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 1

Linhas no Treino: 462121


26/05/10 17:36:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:36:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Linhas no Teste: 115396


In [180]:
def evaluate_model(model_name: str, predictions) -> None:
    predictions = predictions.cache()
    predictions.createOrReplaceTempView("predictions")

    rmse = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="rmse",
    ).evaluate(predictions)
    mae = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="mae",
    ).evaluate(predictions)
    r2 = RegressionEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol="prediction",
        metricName="r2",
    ).evaluate(predictions)

    print(f"\n{model_name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2: {r2:.6f}")

    predictions.unpersist()

In [181]:
for model_name, estimator in models:
    model = estimator.fit(train_prepared)
    predictions = model.transform(test_prepared)
    evaluate_model(model_name, predictions)

26/05/10 17:37:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 1


Linear Regression
RMSE: 240.41
MAE: 180.02
R2: 0.290118


26/05/10 17:37:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 17:37:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 1


Decision Tree Regressor - tuned depth 12
RMSE: 137.32
MAE: 84.57
R2: 0.768403


26/05/10 17:37:22 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/05/10 17:37:22 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
